# CIFAR T-Test

Run the pairwise T-test notebook after training a model. This notebook clones the repo into Colab if needed and runs the test logic directly in the notebook.

In [3]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/DVBW')
# if not repo_dir.exists():
#     subprocess.run(['git', 'clone', 'https://github.com/THUYimingLi/DVBW.git', str(repo_dir)], check=True)
#     print(f'Cloned repository to {repo_dir}')
# else:
#     print(f'Repository already present at {repo_dir}')

In [4]:
from pathlib import Path
import os

candidates = [Path.cwd(), Path.cwd() / 'CIFAR', Path('/content/DVBW/CIFAR'), Path('/content/drive/MyDrive/DVBW/CIFAR')]
WORKDIR = next((path for path in candidates if (path / 'model.py').exists()), None)
if WORKDIR is None:
    raise FileNotFoundError('Could not find the CIFAR folder. Set WORKDIR manually if needed.')
os.chdir(WORKDIR)
print(f'Working directory: {WORKDIR}')

Working directory: c:\Users\emrea\Documents\Uni\SPML\DVBW\CIFAR


In [5]:
%pip install -q scipy pillow numpy

Note: you may need to restart the kernel to use updated packages.


In [6]:
MODEL = 'resnet'  # 'resnet' or 'vgg'
MODEL_PATH = './checkpoint/infected/resnet_noise/model_best_005.pth.tar'
# MODEL_PATH = './checkpoint/infected/vgg_checkered/model_best.pth.tar'
CLEAN_MODEL_PATH = './checkpoint/benign/resnet/model_best.pth.tar'
TRIGGER_PATH = './triggers/hf_noise_trigger_32x32.png'
ALPHA_PATH = './triggers/alpha_noise_005.png'
TARGET_LABEL = 0
NUM_IMG = 100
TEST_BATCH = 16
WORKERS = 2
GPU_ID = '0'
MARGIN = 0.2
SEED = 666

In [7]:
import os
import random

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from PIL import Image
from scipy.stats import ttest_rel

from model import *
from tools import *

assert MODEL in {'resnet', 'vgg'}
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
use_cuda = torch.cuda.is_available()
if not use_cuda:
    raise RuntimeError('CUDA GPU not available. In Colab, enable a GPU runtime and rerun.')

data_dir = WORKDIR / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
datasets.CIFAR10(root=str(data_dir), train=False, download=True)

another_trigger_path = TRIGGER_PATH.replace('line', 'cross') if 'line' in TRIGGER_PATH else TRIGGER_PATH.replace('cross', 'line')
another_alpha_path = ALPHA_PATH.replace('line', 'cross') if 'line' in ALPHA_PATH else ALPHA_PATH.replace('cross', 'line')

trigger = transforms.ToTensor()(Image.open(TRIGGER_PATH))
alpha = transforms.ToTensor()(Image.open(ALPHA_PATH))
another_trigger = transforms.ToTensor()(Image.open(another_trigger_path))
another_alpha = transforms.ToTensor()(Image.open(another_alpha_path))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f'Using model: {MODEL_PATH}')
print(f'Using clean baseline: {CLEAN_MODEL_PATH}')


def build_model(model_name):
    if model_name == 'resnet':
        return ResNet18()
    return vgg19_bn()


def collect_softmax_outputs(testloader, model):
    model.eval()
    outputs_all = []
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.cuda(), targets.cuda()
            outputs = model(inputs)
            outputs_all += torch.nn.functional.softmax(outputs, dim=1).cpu().numpy().tolist()
    return np.array(outputs_all)


def main():
    main_model = build_model(MODEL)
    clean_model = build_model(MODEL)

    main_checkpoint = torch.load(MODEL_PATH)
    clean_checkpoint = torch.load(CLEAN_MODEL_PATH)

    main_model = torch.nn.DataParallel(main_model).cuda()
    clean_model = torch.nn.DataParallel(clean_model).cuda()
    main_model.load_state_dict(main_checkpoint['state_dict'])
    clean_model.load_state_dict(clean_checkpoint['state_dict'])
    main_model.eval()
    clean_model.eval()
    cudnn.benchmark = True

    transform_test_poisoned = transforms.Compose([TriggerAppending(trigger=trigger, alpha=alpha), transforms.ToTensor()])
    transform_test_another_poisoned = transforms.Compose([TriggerAppending(trigger=another_trigger, alpha=another_alpha), transforms.ToTensor()])
    transform_test_benign = transforms.Compose([transforms.ToTensor()])

    dataloader = datasets.CIFAR10
    test_set_basic = dataloader(root=str(data_dir), train=False, download=True)
    testset_poisoned = dataloader(root=str(data_dir), train=False, download=True, transform=transform_test_poisoned)
    testset_another_poisoned = dataloader(root=str(data_dir), train=False, download=True, transform=transform_test_another_poisoned)
    testset_benign = dataloader(root=str(data_dir), train=False, download=True, transform=transform_test_benign)

    select_img = []
    select_target = []
    for i in range(len(test_set_basic)):
        if test_set_basic.targets[i] != TARGET_LABEL:
            select_img.append(test_set_basic.data[i])
            select_target.append(test_set_basic.targets[i])

    idx = list(np.arange(len(select_img)))
    random.shuffle(idx)
    image_idx = idx[:NUM_IMG]

    testing_img_poisoned = [select_img[i] for i in range(len(select_img)) if i in image_idx]
    testing_img_another_poisoned = [select_img[i] for i in range(len(select_img)) if i in image_idx]
    testing_img_benign = [select_img[i] for i in range(len(select_img)) if i in image_idx]
    testing_target = [select_target[i] for i in range(len(select_img)) if i in image_idx]

    testset_poisoned.data, testset_poisoned.targets = testing_img_poisoned, testing_target
    testset_benign.data, testset_benign.targets = testing_img_benign, testing_target
    testset_another_poisoned.data, testset_another_poisoned.targets = testing_img_another_poisoned, testing_target

    poisoned_loader = torch.utils.data.DataLoader(testset_poisoned, batch_size=TEST_BATCH, shuffle=False, num_workers=WORKERS)
    another_poisoned_loader = torch.utils.data.DataLoader(testset_another_poisoned, batch_size=TEST_BATCH, shuffle=False, num_workers=WORKERS)
    benign_loader = torch.utils.data.DataLoader(testset_benign, batch_size=TEST_BATCH, shuffle=False, num_workers=WORKERS)

    output_main_poisoned = collect_softmax_outputs(poisoned_loader, main_model)
    output_main_benign = collect_softmax_outputs(benign_loader, main_model)
    output_clean_poisoned = collect_softmax_outputs(poisoned_loader, clean_model)
    output_clean_benign = collect_softmax_outputs(benign_loader, clean_model)
    output_another_poisoned = collect_softmax_outputs(another_poisoned_loader, main_model)
    output_another_benign = collect_softmax_outputs(benign_loader, main_model)

    p_main_poisoned = np.array([output_main_poisoned[i, TARGET_LABEL] for i in range(len(output_main_poisoned))])
    p_main_benign = np.array([output_main_benign[i, TARGET_LABEL] for i in range(len(output_main_benign))])
    p_main_another_poisoned = np.array([output_another_poisoned[i, TARGET_LABEL] for i in range(len(output_another_poisoned))])
    p_main_another_benign = np.array([output_another_benign[i, TARGET_LABEL] for i in range(len(output_another_benign))])
    p_clean_poisoned = np.array([output_clean_poisoned[i, TARGET_LABEL] for i in range(len(output_clean_poisoned))])
    p_clean_benign = np.array([output_clean_benign[i, TARGET_LABEL] for i in range(len(output_clean_benign))])

    t_malicious = ttest_rel(p_main_benign + MARGIN, p_main_poisoned, alternative='less')
    t_model_independent = ttest_rel(p_clean_benign + MARGIN, p_clean_poisoned, alternative='less')
    t_trigger_independent = ttest_rel(p_main_another_benign + MARGIN, p_main_another_poisoned, alternative='less')

    path_folder = str(Path(MODEL_PATH).parent)
    print(f'Malicious Ttest p-value: {t_malicious[1]:.4e}, average delta P: {np.mean(p_main_poisoned - p_main_benign):.4e}')
    print(f'Model Independent Ttest p-value: {t_model_independent[1]:.4e}, average delta P: {np.mean(p_clean_poisoned - p_clean_benign):.4e}')
    print(f'Trigger Independent Ttest p-value: {t_trigger_independent[1]:.4e}, average delta P: {np.mean(p_main_another_poisoned - p_main_another_benign):.4e}')

    output_path = Path(path_folder) / f'Ttest_{NUM_IMG}.txt'
    with open(output_path, 'w') as f:
        for i in range(len(p_main_poisoned)):
            f.write('{:04d} {:.4e} {:.4e} {:.4e} {:.4e} {:.4e} {:.4e}\n'.format(image_idx[i], p_main_poisoned[i], p_main_benign[i], p_main_another_poisoned[i], p_main_another_benign[i], p_clean_poisoned[i], p_clean_benign[i]))
        f.write(f'Malicious Ttest p-value: {t_malicious[1]:.4e}, average delta P: {np.mean(p_main_poisoned - p_main_benign):.4e}\n')
        f.write(f'Model Independent Ttest p-value: {t_model_independent[1]:.4e}, average delta P: {np.mean(p_clean_poisoned - p_clean_benign):.4e}\n')
        f.write(f'Trigger Independent Ttest p-value: {t_trigger_independent[1]:.4e}, average delta P: {np.mean(p_main_another_poisoned - p_main_another_benign):.4e}\n')
    print(f'Saved results to {output_path}')


main()

c:\Users\emrea\Documents\Uni\SPML.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Using model: ./checkpoint/infected/resnet_noise/model_best_005.pth.tar
Using clean baseline: ./checkpoint/benign/resnet/model_best.pth.tar
Malicious Ttest p-value: 1.5389e-145, average delta P: 9.9255e-01
Model Independent Ttest p-value: 1.0000e+00, average delta P: -1.8177e-03
Trigger Independent Ttest p-value: 1.5389e-145, average delta P: 9.9255e-01
Saved results to checkpoint\infected\resnet_noise\Ttest_100.txt
